In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from utils import feature_engineering

In [2]:
data = pd.read_csv('heart.csv')

In [3]:
data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [4]:
oe = OrdinalEncoder()

data['ST_Slope'] = oe.fit_transform(data[['ST_Slope']])
print(data['ST_Slope'].value_counts())
print(oe.categories_)

ST_Slope
1.0    460
2.0    395
0.0     63
Name: count, dtype: int64
[array(['Down', 'Flat', 'Up'], dtype=object)]


In [5]:
data['ST_Slope'].head()

0    2.0
1    1.0
2    2.0
3    1.0
4    2.0
Name: ST_Slope, dtype: float64

In [6]:
le = LabelEncoder()

data['ExerciseAngina'] = le.fit_transform(data['ExerciseAngina'])
print(data['ExerciseAngina'].value_counts())
print(le.classes_)

ExerciseAngina
0    547
1    371
Name: count, dtype: int64
['N' 'Y']


In [7]:
data['Sex'] = le.fit_transform(data['Sex'])
print(data['Sex'].value_counts())
print(le.classes_)

Sex
1    725
0    193
Name: count, dtype: int64
['F' 'M']


In [8]:
chest_pain_dummies = pd.get_dummies(data['ChestPainType'], prefix='ChestPain')
resting_ecg_dummies = pd.get_dummies(data['RestingECG'], prefix='RestingECG')

data = pd.concat([data, chest_pain_dummies, resting_ecg_dummies], axis=1)

if 'ChestPainType' in data.columns:
    data = data.drop('ChestPainType', axis=1)
if 'RestingECG' in data.columns:
    data = data.drop('RestingECG', axis=1)

print('Added ChestPain dummy columns:', list(chest_pain_dummies.columns))
print('Added RestingECG dummy columns:', list(resting_ecg_dummies.columns))
print('New data shape:', data.shape)

Added ChestPain dummy columns: ['ChestPain_ASY', 'ChestPain_ATA', 'ChestPain_NAP', 'ChestPain_TA']
Added RestingECG dummy columns: ['RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST']
New data shape: (918, 17)


In [9]:
data.columns

Index(['Age', 'Sex', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR',
       'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease',
       'ChestPain_ASY', 'ChestPain_ATA', 'ChestPain_NAP', 'ChestPain_TA',
       'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST'],
      dtype='object')

In [10]:
data.to_csv('heart_preprocessed.csv', index=False)

In [11]:
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data.drop('HeartDisease', axis=1))
new_scaled_data = pd.DataFrame(data_scaled, columns=data.columns[:-1])
new_scaled_data['HeartDisease'] = data['HeartDisease']

In [12]:
new_scaled_data.to_csv('heart_preprocessed_scaled.csv', index=False)

In [13]:
data = feature_engineering(data)

In [14]:
data.columns

Index(['Age', 'Sex', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR',
       'ExerciseAngina', 'Oldpeak', 'ST_Slope', 'HeartDisease',
       'ChestPain_ASY', 'ChestPain_ATA', 'ChestPain_NAP', 'ChestPain_TA',
       'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST',
       'Cholesterol_log', 'Oldpeak_log', 'Age_bin', 'BP_category',
       'Chol_Age_ratio', 'HR_per_Age'],
      dtype='object')

In [15]:
# Numerical features (original + engineered ratios/logs)
num_features = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak",
                "Cholesterol_log", "Oldpeak_log", "Chol_Age_ratio", "HR_per_Age"]

# Categorical features (engineered categories)
cat_features = ["Age_bin", "BP_category"]

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(), cat_features)  
    ]
)

In [17]:
pipeline = Pipeline(steps=[("preprocessor", preprocessor)])

In [18]:
processed = pipeline.fit_transform(data)

processed_df = pd.DataFrame(
    processed.toarray() if hasattr(processed, "toarray") else processed,
    columns=(num_features + list(pipeline.named_steps["preprocessor"]
                                 .transformers_[1][1]
                                 .get_feature_names_out(cat_features)))
)

In [19]:
processed_df.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Cholesterol_log,Oldpeak_log,Chol_Age_ratio,HR_per_Age,Age_bin_Elderly,Age_bin_Middle,Age_bin_Young,BP_category_Hypertension,BP_category_Normal,BP_category_PreHypertension
0,-1.433140,0.463654,0.886771,-0.551341,1.384080,-0.851276,0.711377,-0.873474,1.591958,1.823629,0.0,1.0,0.0,1.0,0.0,0.0
1,-0.478484,1.641229,-0.250184,-0.551341,0.754610,0.118532,0.115591,0.220708,-0.147695,0.563288,0.0,1.0,0.0,1.0,0.0,0.0
2,-1.751359,-0.125133,0.824187,-0.551341,-1.527219,-0.851276,0.684953,-0.873474,1.799475,-0.040759,0.0,0.0,1.0,0.0,0.0,1.0
3,-0.584556,0.345897,0.104463,-0.551341,-1.133801,0.603436,0.333161,0.691770,0.236756,-0.490837,0.0,1.0,0.0,0.0,0.0,1.0
4,0.051881,1.052442,-0.093722,-0.551341,-0.583014,-0.851276,0.216221,-0.873474,-0.178240,-0.480383,0.0,1.0,0.0,1.0,0.0,0.0


In [20]:
processed_df.columns

Index(['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak',
       'Cholesterol_log', 'Oldpeak_log', 'Chol_Age_ratio', 'HR_per_Age',
       'Age_bin_Elderly', 'Age_bin_Middle', 'Age_bin_Young',
       'BP_category_Hypertension', 'BP_category_Normal',
       'BP_category_PreHypertension'],
      dtype='object')

In [21]:
final_data = pd.concat([processed_df, data[['ST_Slope','ExerciseAngina','ChestPain_ASY', 'ChestPain_ATA', 'ChestPain_NAP', 'ChestPain_TA',
       'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST','HeartDisease']]], axis=1)

In [22]:
final_data.columns

Index(['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak',
       'Cholesterol_log', 'Oldpeak_log', 'Chol_Age_ratio', 'HR_per_Age',
       'Age_bin_Elderly', 'Age_bin_Middle', 'Age_bin_Young',
       'BP_category_Hypertension', 'BP_category_Normal',
       'BP_category_PreHypertension', 'ST_Slope', 'ExerciseAngina',
       'ChestPain_ASY', 'ChestPain_ATA', 'ChestPain_NAP', 'ChestPain_TA',
       'RestingECG_LVH', 'RestingECG_Normal', 'RestingECG_ST', 'HeartDisease'],
      dtype='object')

In [23]:
final_data.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Cholesterol_log,Oldpeak_log,Chol_Age_ratio,HR_per_Age,...,ST_Slope,ExerciseAngina,ChestPain_ASY,ChestPain_ATA,ChestPain_NAP,ChestPain_TA,RestingECG_LVH,RestingECG_Normal,RestingECG_ST,HeartDisease
0,-1.433140,0.463654,0.886771,-0.551341,1.384080,-0.851276,0.711377,-0.873474,1.591958,1.823629,...,2.0,0,False,True,False,False,False,True,False,0
1,-0.478484,1.641229,-0.250184,-0.551341,0.754610,0.118532,0.115591,0.220708,-0.147695,0.563288,...,1.0,0,False,False,True,False,False,True,False,1
2,-1.751359,-0.125133,0.824187,-0.551341,-1.527219,-0.851276,0.684953,-0.873474,1.799475,-0.040759,...,2.0,0,False,True,False,False,False,False,True,0
3,-0.584556,0.345897,0.104463,-0.551341,-1.133801,0.603436,0.333161,0.691770,0.236756,-0.490837,...,1.0,1,True,False,False,False,False,True,False,1
4,0.051881,1.052442,-0.093722,-0.551341,-0.583014,-0.851276,0.216221,-0.873474,-0.178240,-0.480383,...,2.0,0,False,False,True,False,False,True,False,0


In [24]:
round(final_data['HeartDisease'].value_counts()/len(final_data) * 100,2)

HeartDisease
1    55.34
0    44.66
Name: count, dtype: float64

In [25]:
final_data.to_csv('preprocessed_heart.csv', index=False)